# Neural language model (naive implementation)

À rendre pour le 10 février (mail "MIDS TP2" avec le lien vers votre colab, avec droits éditeur).

Vous implémenterez un modèle de langue neuronal simple, qui:
- prend en entrée les k=4 mots précédents la position i
- et fournit les log-probabilités de chaque mot du vocabulaire d'apparaître à cette position.


Le réseau utilisera la **concaténation d'embeddings** des k mots d'entrée, suivie d'un MLP à une couche cachée.

In [99]:
# Inspired by
# https://pytorch.org/tutorials/beginner/nlp/word_embeddings_tutorial.html
# by Robert Guthrie


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

## TODO1 : build the training and testing tensors

In [100]:
# Tiny training corpus: we will use a French poem by Victor Hugo
# (To tokenize it just split on spaces)
train_sent = """
Demain , dès l’ aube, à l’ heure où blanchit la campagne, je partirai .
Vois -tu , je sais que tu m’ attends .
J’ irai par la forêt , j’ irai par la montagne .
Je ne puis demeurer loin de toi plus longtemps .
Je marcherai les yeux fixés sur mes pensées ,
Sans rien voir au dehors , sans entendre aucun bruit ,
Seul , inconnu , le dos courbé , les mains croisées , triste, et le jour pour moi sera comme la nuit .
Je ne regarderai ni l’ or du soir qui tombe , ni les voiles au loin descendant vers Harfleur ,
Et quand j’ arriverai, je mettrai sur ta tombe un bouquet de houx vert et de bruyère en fleur .
""".split()

# TODO: Build the list of examples,
# each example being a pair (context, target)
# namely:  ([ word_i-k,..., word_i-2, word_i-1 ], target word)

def build_examples(k):
    train_examples = []
    for i in range(k, len(train_sent) - k):
        context = train_sent[i - k : i]
        target = train_sent[i]
        train_examples.append((context, target))
    return train_examples

train_examples = build_examples(k=4)
# check by printing the first 3 examples
print(train_examples[:3])

vocab = set(train_sent)
w2i = {word: i for i, word in enumerate(vocab)}

# TODO: encode the training examples into word ids
# (better to do it once, instead of within the training loops)
train_examples =  [([w2i[word] for word in context], w2i[target]) for context, target in train_examples]
print(train_examples[:3])
print(len(vocab))

[(['Demain', ',', 'dès', 'l’'], 'aube,'), ([',', 'dès', 'l’', 'aube,'], 'à'), (['dès', 'l’', 'aube,', 'à'], 'l’')]
[([73, 49, 40, 36], 78), ([49, 40, 36, 78], 33), ([40, 36, 78, 33], 36)]
90


## TODO2: fill in the \_\_init\_\_ and \_\_forward\_\_ method of the network

In [101]:
class MyLanguageModel(nn.Module):

    def __init__(self, vocab_size, embedding_size, context_size, hidden_layer_size):
        super(MyLanguageModel, self).__init__()
        # the target network should work as follows:
        #  for a single input
        # - input =  context_size word ids (context words)
        #    => retrieval of their embeddings
        #    => concatenation (see the forward method below)
        # - which provides the "embedding layer"
        #    => linear combination
        #    => reLU
        # - which provides the "hidden layer"
        #    => linear combination
        #    => log_softmax
        # - which provides log probabilities over the full vocabulary

        # but remember nn.Module works with a batch of inputs, not a single input

        # TODO: declare the appropriate members for this nn.Module (everything that will act as parameter tensor)

        #Definition of the embedding layer
        self.embeddings = nn.Embedding(vocab_size, embedding_size)
        #Definition of the hidden layer
        self.linear1 = nn.Linear(embedding_size * context_size, hidden_layer_size)
        #Definition of the output layer
        self.linear2 = nn.Linear(hidden_layer_size, vocab_size)
        # No need to declare the ReLU and log_softmax as members, we can just use the functions in the forward method



    def forward(self, inputs):
        size = inputs.size()              # [BATCH_SIZE, CONTEXT_SIZE]


        # TODO: forward propagation
        #  (cf. description above of what the network should do)
        #  to output log probabilities,

        # NB: for each intermediate tensor,
        #     write its shape as a comment
        out = self.embeddings(inputs)         # [BATCH_SIZE, CONTEXT_SIZE, EMBEDDING_SIZE]
        out = out.view(size[0], -1)          # [BATCH_SIZE, CONTEXT_SIZE * EMBEDDING_SIZE]
        out = self.linear1(out)              # [BATCH_SIZE, HIDDEN_LAYER_SIZE]
        out = F.relu(out)                    # [BATCH_SIZE, HIDDEN_LAYER_SIZE]
        out = self.linear2(out)              # [BATCH_SIZE, VOCAB_SIZE]
        log_probs = F.log_softmax(out, dim=1)  # [BATCH_SIZE, VOCAB_SIZE]
        return log_probs





In [102]:
CONTEXT_SIZE = 4 # size of the context window (i.e. how many previous words to look at)
EMB_SIZE = 10
HIDDEN_LAYER_SIZE = 50

# instance of MyLanguageModel
my_language_model = MyLanguageModel(len(vocab), EMB_SIZE, CONTEXT_SIZE, HIDDEN_LAYER_SIZE)


## TODO3 : training

In [103]:

# to store the training losses at each epoch
train_losses = []
loss_function = nn.NLLLoss()

# the optimizer is the instance that will actually update the declared parameters
optimizer = optim.SGD(my_language_model.parameters(), lr=0.05)

from random import shuffle

NB_EPOCHS = 20
BATCH_SIZE = 5

def epoch_training():
    for epoch in range(NB_EPOCHS):
        epoch_loss = 0

        # shuffle data
        shuffle(train_examples)

        # TODO loop on batches
        #      and training step for a batch
        # Create batches manually
        for batch_idx in range(0, len(train_examples), BATCH_SIZE):
            batch = train_examples[batch_idx : batch_idx + BATCH_SIZE]
            # prepare the batch data
            batch_contexts = [context for context, target in batch]
            batch_targets = [target for context, target in batch]
            # Convert to tensors
            batch_contexts = torch.LongTensor(batch_contexts)  # [BATCH_SIZE, CONTEXT_SIZE]
            batch_targets = torch.LongTensor(batch_targets)  # [BATCH_SIZE]
            # zero the parameter gradients
            my_language_model.zero_grad()
            # forward + backward + optimize
            log_probs = my_language_model(batch_contexts)  # [BATCH_SIZE, VOCAB_SIZE]
            loss = loss_function(log_probs, batch_targets)
            loss.backward()
            optimizer.step()    
            epoch_loss += loss.item()

        train_losses.append(epoch_loss)
epoch_training()
print(train_losses)


[112.63538980484009, 108.38367652893066, 104.12988114356995, 99.80604219436646, 95.42112636566162, 91.10459780693054, 86.90741729736328, 81.39062452316284, 75.88771748542786, 70.79936909675598, 64.53686261177063, 58.106860280036926, 52.83411955833435, 45.96956765651703, 40.60574680566788, 33.96646934747696, 28.22841578722, 23.613906979560852, 19.51116779446602, 16.063443809747696]


## TODO4: implement a text generator

Greedily sample words using the model to generate a text of 100 words.

In [104]:
# Create index to word mapping
i2w = {i: word for word, i in w2i.items()}

# Text generation: greedy sampling
def generate_text(model, context_ids, num_words, context_size,training_samples):
    """
    Generate text greedily using the trained language model.
    
    Args:
        model: trained MyLanguageModel
        context_ids: initial context as list of word ids (length = context_size)
        num_words: number of words to generate
        context_size: size of context window
    
    Returns:
        generated_text: list of generated word ids
    """
    # Create a mapping from context to target for accuracy evaluation
    context_to_target = {tuple(ctx): tgt for (ctx, tgt) in training_samples}
    generated = list(context_ids)  # start with initial context
    gold_labels = []
    pred_labels = []
    with torch.no_grad():  # no need to compute gradients during generation
        for _ in range(num_words):
            # Keep only the last context_size words
            current_context = generated[-context_size:]
            gold_labels.append(context_to_target.get(tuple(current_context), None))
            # Convert to tenso
            context_tensor = torch.LongTensor([current_context])  # [1, CONTEXT_SIZE]
            
            # Get log probabilities from model
            log_probs = model(context_tensor)  # [1, VOCAB_SIZE]
            
            # Greedy: take the word with highest probability
            next_word_id = torch.argmax(log_probs[0]).item()
            pred_labels.append(next_word_id)
            
            # Add to generated text
            generated.append(next_word_id)
        # Compute accuracy
        valid = [(g, p) for g, p in zip(gold_labels, pred_labels) if g is not None]
        if len(valid) == 0:
            accuracy = float("nan")
            print("Generation Accuracy: N/A (aucun contexte généré ne correspond aux training_samples)")
        else:
            accuracy = sum(1 for g, p in valid if g == p) / len(valid)
            print("Generation Accuracy:", accuracy)
    
    return generated




In [105]:
# Initialize context with the first context of the training data
initial_context = list(train_examples[0][0])

print("Initial context (ids):", initial_context)
print("Initial context (words):", [i2w[wid] for wid in initial_context])

# Generate 100 words
generated_ids = generate_text(my_language_model, initial_context,96, 
                              context_size=CONTEXT_SIZE,training_samples=train_examples)

# Convert ids to words
generated_text = [i2w[wid] for wid in generated_ids]

print("\nGenerated text:")
print(" ".join(generated_text))
print(len(generated_text))

Initial context (ids): [49, 30, 37, 89]
Initial context (words): [',', 'ni', 'les', 'voiles']
Generation Accuracy: 0.9880952380952381

Generated text:
, ni les voiles au loin descendant vers Harfleur , Et quand j’ arriverai, je mettrai sur ta tombe un bouquet de houx vert et de sur , je sais que tu m’ attends . J’ la , Seul , inconnu , le dos courbé , les mains croisées , triste, et le jour pour moi sera comme la nuit . Je ne regarderai ni l’ or du soir qui tombe , ni les voiles au loin descendant vers Harfleur , Et quand j’ arriverai, je mettrai sur ta tombe un bouquet de houx vert et de sur , je
100


## TODO5 : test on a bigger corpus

In [106]:
import nltk
nltk.download('punkt_tab') # needed to use nltk.tokenize.sent_tokenize

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/fatoumata/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [107]:
sample_text = """M. Myriel devait subir le sort de tout nouveau venu dans une petite ville où il y a beaucoup de bouches qui parlent et fort peu de têtes qui pensent. Il devait le subir, quoiqu’il fût évêque et parce qu’il était évêque. Mais, après tout, les propos auxquels on mêlait son nom n’étaient peut-être que des propos ; du bruit, des mots, des paroles ; moins que des paroles, des palabres, comme dit l’énergique langue du midi.

Quoi qu’il en fût, après neuf ans d’épiscopat et de résidence à Digne, tous ces racontages, sujets de conversation qui occupent dans le premier moment les petites villes et les petites gens, étaient tombés dans un oubli profond. Personne n’eût osé en parler, personne n’eût même osé s’en souvenir."""
# la segmentation en phrases peut se faire avec `sent_tokenize`, issu d'un apprentissage automatique
sentences = nltk.tokenize.sent_tokenize(sample_text)
print(sentences)

['M. Myriel devait subir le sort de tout nouveau venu dans une petite ville où il y a beaucoup de bouches qui parlent et fort peu de têtes qui pensent.', 'Il devait le subir, quoiqu’il fût évêque et parce qu’il était évêque.', 'Mais, après tout, les propos auxquels on mêlait son nom n’étaient peut-être que des propos ; du bruit, des mots, des paroles ; moins que des paroles, des palabres, comme dit l’énergique langue du midi.', 'Quoi qu’il en fût, après neuf ans d’épiscopat et de résidence à Digne, tous ces racontages, sujets de conversation qui occupent dans le premier moment les petites villes et les petites gens, étaient tombés dans un oubli profond.', 'Personne n’eût osé en parler, personne n’eût même osé s’en souvenir.']


In [108]:
# si on suppose que les retours à la ligne correspondent à des frontières de phrase
# (attention ce n'est pas toujours le cas)

def split_into_sentences(text):
    sent_lists = [nltk.tokenize.sent_tokenize(x) for x in text.split('\n')]
    return [sent.strip() for sent_list in sent_lists for sent in sent_list]

sentences2 = split_into_sentences(sample_text)
print(sentences == sentences2)

True


### Récupération du texte des romans

In [109]:
%%capture

import requests  # to download web files

!pip install --upgrade beautifulsoup4 # to parse HTML files
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

In [89]:
book_urls = [
    # Les misérables
    'https://www.ebooksgratuits.com/html/hugo_les_miserables_fantine.html',
    'https://www.ebooksgratuits.com/html/hugo_les_miserables_cosette.html',
    'https://www.ebooksgratuits.com/html/hugo_les_miserables_marius.html',
    'https://www.ebooksgratuits.com/html/hugo_les_miserables_idylle_plumet_epopee_st_denis.html',
    'https://www.ebooksgratuits.com/html/hugo_les_miserables_jean_valjean.html'
]


In [110]:
%%time

pages = []
for url in tqdm(book_urls):
  page = requests.get(url)
  pages.append(page)

100%|██████████| 5/5 [00:03<00:00,  1.36it/s]

CPU times: user 199 ms, sys: 25.9 ms, total: 225 ms
Wall time: 3.68 s


On récupère le texte au format HTML à l'aide de la librairie Beautiful Soup, plus précisément le contenu des balises \<p\>.


In [111]:
all_sentences = []

for page in pages:
  soup = BeautifulSoup(page.content, 'html.parser') # using the python built-in HTML parser
  #print(soup.prettify()) # to have a look at the HTML

  # gutenberg files have bare <p> sections, so no restriction
  paragraphs = soup.find_all('p')
  # see https://beautiful-soup-4.readthedocs.io/en/latest/
  # for the arguments of get_text
  # here: the content of <p> sections will be separated by \n
  paragraphs = [p.get_text('\n', strip=True) for p in paragraphs]

  # splitting on all white-space like characters, and then reconcatenating using a single space separator
  paragraphs = [' '.join(p.split()) for p in paragraphs]
  paragraphs = [nltk.tokenize.sent_tokenize(p) for p in paragraphs]
  sentences = [sent for p in paragraphs for sent in p]
  print(sentences[:100])
  all_sentences.extend(sentences)

['Victor Hugo', 'LES MISÉRABLES Tome I – FANTINE', 'Texte annoté par Guy Rosa, professeur à l’Université Paris-Diderot 1862', 'Table des matières', 'Livre premier Un juste .', '6', 'Chapitre I Monsieur Myriel 7', 'Chapitre II Monsieur Myriel devient monseigneur Bienvenu .. 11', 'Chapitre III À bon évêque dur évêché .', '19', 'Chapitre IV Les œuvres semblables aux paroles .', '22', 'Chapitre V Que monseigneur Bienvenu faisait durer trop longtemps ses soutanes 32', 'Chapitre VI Par qui il faisait garder sa maison .. 37', 'Chapitre VII Cravatte .', '45', 'Chapitre VIII Philosophie après boire .', '51', 'Chapitre IX Le frère raconté par la sœur .', '56', 'Chapitre X L’évêque en présence d’une lumière inconnue .', '61', 'Chapitre XI Une restriction .. 79', 'Chapitre XII Solitude de monseigneur Bienvenu .. 85', 'Chapitre XIII Ce qu’il croyait .', '89', 'Chapitre XIV Ce qu’il pensait .', '95', 'Livre deuxième La chute .', '99', 'Chapitre I Le soir d’un jour de marche .', '100', 'Chapitre II L

In [112]:
print("Downloaded {:,d} books for a total of {:,d} paragraphs.".format(len(book_urls), len(paragraphs)))

Downloaded 5 books for a total of 2,756 paragraphs.


### Tokenization du corpus de romans

In [113]:
from collections import Counter


In [114]:
tokenized_corpus = []

# on rajoute un symbole spécial au début de chaque phrase
# (servant aussi de fin de la phrase précédente)
for s in all_sentences:
    tokenized_corpus.extend(['*BOS*'] + nltk.tokenize.word_tokenize(s, language='french'))

print("Extracted {:,d} tokens".format(len(tokenized_corpus)))
print(tokenized_corpus[10000:10100])

Extracted 759,060 tokens
['il', 'tombait', 'tout', 'à', 'coup', 'dans', 'une', 'méditation', 'profonde', ',', 'd', '’', 'où', 'il', 'ne', 'sortait', 'que', 'pour', 'écrire', 'quelques', 'lignes', 'sur', 'les', 'pages', 'mêmes', 'du', 'volume', '.', '*BOS*', 'Ces', 'lignes', 'souvent', 'n', '’', 'ont', 'aucun', 'rapport', 'avec', 'le', 'livre', 'qui', 'les', 'contient', '.', '*BOS*', 'Nous', 'avons', 'sous', 'les', 'yeux', 'une', 'note', 'écrite', 'par', 'lui', 'sur', 'une', 'des', 'marges', 'd', '’', 'un', 'in-quarto', 'intitulé', ':', 'Correspondance', 'du', 'lord', 'Germain', 'avec', 'les', 'généraux', 'Clinton', ',', 'Cornwallis', 'et', 'les', 'amiraux', 'de', 'la', 'station', 'de', 'l', '’', 'Amérique', '.', '*BOS*', 'À', 'Versailles', ',', 'chez', 'Poinçot', ',', 'libraire', ',', 'et', 'à', 'Paris', ',', 'chez']


In [115]:
train_sent = tokenized_corpus
train_examples = build_examples(k=4)
# check by printing the first 3 examples
print(train_examples[:3])

vocab = set(train_sent)
w2i = {word: i for i, word in enumerate(vocab)}

# encode the training examples into word ids
train_examples =  [([w2i[word] for word in context], w2i[target]) for context, target in train_examples]
print(train_examples[:3])
print(len(vocab))


[(['*BOS*', 'Victor', 'Hugo', '*BOS*'], 'LES'), (['Victor', 'Hugo', '*BOS*', 'LES'], 'MISÉRABLES'), (['Hugo', '*BOS*', 'LES', 'MISÉRABLES'], 'Tome')]
[([14265, 33935, 6540, 14265], 21070), ([33935, 6540, 14265, 21070], 1004), ([6540, 14265, 21070, 1004], 32824)]
34903


In [116]:
# instance of MyLanguageModel
my_language_model = MyLanguageModel(len(vocab), EMB_SIZE, CONTEXT_SIZE, HIDDEN_LAYER_SIZE)

In [117]:
# training of the model on de bigger corpus
train_losses = []
NB_EPOCHS = 5
BATCH_SIZE = 256
epoch_training()
print(train_losses)


[31144.655660629272, 31144.475379943848, 31144.531336784363, 31144.53924179077, 31144.579251289368]


In [118]:
# Create index to word mapping
i2w = {i: word for word, i in w2i.items()}
# Initialize context with the first context of the training data
initial_context = list(train_examples[0][0])

print("Initial context (ids):", initial_context)
print("Initial context (words):", [i2w[wid] for wid in initial_context])

# Generate 100 words
generated_ids = generate_text(my_language_model, initial_context,96, 
                              context_size=CONTEXT_SIZE,training_samples=train_examples)

# Convert ids to words
generated_text = [i2w[wid] for wid in generated_ids]

print("\nGenerated text:")
print(" ".join(generated_text))
print(len(generated_text))

Initial context (ids): [21666, 5783, 16600, 2634]
Initial context (words): ['’', 'il', 'travaillait', 'à']
Generation Accuracy: 0.0

Generated text:
’ il travaillait à professions épaisse Thrasybule entamant Punissons trouer iras rattacha Saint-Maur insupportables ode Parvenus renaîtra versants Loire auditoire Chonodemaire allaitant inhabité constituait camper divorce portières blanchies Humann truands bâtir camper Rendez haridelle Thrasybule furent abandonnons prestige engorgés Ève Bologne recevait loges Ce pentures Auprès expirant Ensuite tas Maitland bâtir camper Rendez allaitant Thrasybule entamant humour furent 25 constituait Lézardes entrailles murés brioches Thrasybule angle congé Lézardes dénudation Spartiate Madelonnettes Thrasybule furent pyramidal libérateurs remettrai Loire Normand Lézardes véridique usurpe Lézardes contrebas miracolo émasculer entre-croisement Ensuite congé dénoncées insupportables atteignant n'avaient ruisselants moisies Marius… entendons-nous complot ret

# Remarque
Lorsque la taille du contexte k est faible et que l’architecture du modèle est simple (MLP), le modèle de langage ne parvient pas à modéliser correctement les dépendances longues du texte.
Les prédictions restent localement cohérentes mais la génération globale devient rapidement incohérente.